1. How it handles "All Operations"
It is important to understand that "Mixed Precision" does not mean float16 for everything. It is a strategic split:

Computations (Matrix Multiplications, Convolutions): These are done in float16. This is what gives you the speedup on your Colab GPU (T4/A100).

Variables (Weights, Biases): These are stored in float32. This prevents "rounding to zero" during small weight updates, ensuring your model actually learns.

Master Loss Scaling: Because you are using mixed_float16, Keras will automatically apply Loss Scaling. This prevents the gradients from becoming so small that they vanish in float16 (a common issue called underflow).

2. Critical Observation: The "Weight Tying" in your SLM
In your GPT class, you have this specific block:

`logits = tf.matmul(x, self.token_emb.embeddings, transpose_b=True)
return tf.cast(logits, tf.float32)`

3. The MatMul: Since your policy is mixed_float16, the x (activations) will be float16. Keras will automatically cast the self.token_emb.embeddings (which are stored in float32) to float16 for this multiplication.

4. The Final Cast: You correctly cast to tf.float32 at the end. This is a best practice for numerical stability in the Loss function (Softmax).

5. One Tip for your SLM: If you see the model's loss becoming NaN early in training, it is likely because a value inside the Transformer blocks exceeded 65,504 (the maximum value for float16). If that happens, ensure your LayerNormalization epsilon is not too small and your initialization is scaled properly.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================
# Small GPT Language Model (Optimized for Colab GPU)
# Mixed Precision + Cosine Decay + Checkpoints
# ============================================

!pip install datasets tiktoken --quiet

import os
import math
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.utils import plot_model
from tensorflow.keras import layers
from tensorflow.keras import mixed_precision
import matplotlib.pyplot as plt
import tiktoken
from datasets import load_dataset
from tqdm.auto import tqdm


# ============================================
# Enable GPU + Mixed Precision
# ============================================

print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

# Enable memory growth (important in Colab)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)

# 🔥 Enable Mixed Precision (FP16)
mixed_precision.set_global_policy("mixed_float16")

print("Mixed Precision Policy:", mixed_precision.global_policy())


import shutil

# 1. Define paths
DRIVE_PATH = "/content/drive/MyDrive/SLM_Project"
local_train = "train.bin"
local_val = "validation.bin"
drive_train = f"{DRIVE_PATH}/train.bin"
drive_val = f"{DRIVE_PATH}/validation.bin"

# 2. Dataset Management Logic
def prepare_dataset():
    # Check if local files exist; if not, check Drive
    for local_file, drive_file in [(local_train, drive_train), (local_val, drive_val)]:
        if not os.path.exists(local_file):
            if os.path.exists(drive_file):
                print(f"Copying {local_file} from Google Drive...")
                shutil.copy(drive_file, local_file)
            else:
                print(f"Warning: {local_file} not found locally or in Drive. It will be generated.")

prepare_dataset()

if not os.path.exists(drive_train):
    print("First-time run: Backing up datasets to Google Drive...")

    # ============================================
    # Load Dataset
    # ============================================

    ds = load_dataset("roneneldan/TinyStories")
    enc = tiktoken.get_encoding("gpt2")

    # ============================================
    # Tokenization
    # ============================================

    def process(example):
        # ids = enc.encode_ordinary(example["text"])
        ids = enc.encode(example["text"])
        return {"ids": ids, "len": len(ids)}

    if not os.path.exists("train.bin"):
        tokenized = ds.map(
            process,
            remove_columns=["text"],
            num_proc=8,
            desc="Tokenizing"
        )

        for split, dset in tokenized.items():
            arr_len = np.sum(dset["len"], dtype=np.uint64)
            filename = f"{split}.bin"
            arr = np.memmap(filename, dtype=np.uint16, mode="w+", shape=(arr_len,))
            idx = 0

            for sample in tqdm(dset):
                ids = np.array(sample["ids"], dtype=np.uint16)
                arr[idx:idx + len(ids)] = ids
                idx += len(ids)

            arr.flush()

# Ensure the destination directory exists on Google Drive
os.makedirs(DRIVE_PATH, exist_ok=True) # <--- ADD THIS LINE

# Copy *.bin files to Drive for re-use
print("Backing up datasets to Google Drive...")
shutil.copy(local_train, drive_train)
shutil.copy(local_val, drive_val)
print("Backup complete.")

# ============================================
# tf.data Pipeline (GPU Optimized)
# ============================================

block_size = 128
batch_size = 32   # Safe for Colab T4 with FP16

def get_dataset(split):
    filename = "train.bin" if split == "train" else "validation.bin"
    data = np.memmap(filename, dtype=np.uint16, mode="r")

    # Calculate how many full blocks we have
    num_samples = len(data) // (block_size + 1)

    def generator():
        # Iterate through the data linearly instead of randomly
        # This ensures every token is seen exactly once per epoch
        for i in range(0, len(data) - block_size - 1, block_size):
            x = np.array(data[i : i + block_size], dtype=np.int32)
            y = np.array(data[i + 1 : i + 1 + block_size], dtype=np.int32)
            yield x, y

    return (
        tf.data.Dataset.from_generator(
            generator,
            output_signature=(
                tf.TensorSpec((block_size,), tf.int32),
                tf.TensorSpec((block_size,), tf.int32),
            ),
        )
        .shuffle(10000) # Shuffle to maintain randomness
        .batch(batch_size, drop_remainder=True)
        .prefetch(tf.data.AUTOTUNE)
    )

train_ds = get_dataset("train")
val_ds = get_dataset("validation")

# ============================================
# Transformer Block
# ============================================

def causal_mask(seq_len):
    return tf.linalg.band_part(tf.ones((seq_len, seq_len)), -1, 0)

class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        self.att = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            dropout=dropout
        )
        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(4 * embed_dim, activation="gelu"),
            tf.keras.layers.Dense(embed_dim),
        ])
        self.ln1 = tf.keras.layers.LayerNormalization(epsilon=1e-5)
        self.ln2 = tf.keras.layers.LayerNormalization(epsilon=1e-5)
        self.dropout1 = tf.keras.layers.Dropout(dropout)
        self.dropout2 = tf.keras.layers.Dropout(dropout)

    def call(self, x, training=False):
        seq_len = tf.shape(x)[1]
        mask = causal_mask(seq_len)

        attn_output = self.att(x, x, attention_mask=mask, training=training)
        # Post-LN: OK
        x = self.ln1(x + self.dropout1(attn_output, training=training))
        # Pre-LN: Gpt2 or Gpt3 style
        # x = x + self.dropout1(self.att(self.ln1(x)))
        ffn_output = self.ffn(x)
        return self.ln2(x + self.dropout2(ffn_output, training=training))

# ============================================
# GPT Model with Weight Tying
# ============================================

class GPT(tf.keras.Model):
    def __init__(self, vocab_size, block_size,
                 n_layer=6, n_head=4, n_embd=256, dropout=0.1, **kwargs):

        # 1. Pass kwargs to super() to handle standard Keras arguments
        super().__init__(**kwargs)

        # 2. Save these as attributes so get_config can find them later
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.n_layer = n_layer
        self.n_head = n_head
        self.n_embd = n_embd
        self.dropout = dropout

        self.token_emb = tf.keras.layers.Embedding(vocab_size, n_embd)
        self.pos_emb = tf.keras.layers.Embedding(block_size, n_embd)
        self.drop = tf.keras.layers.Dropout(dropout)

        self.blocks = tf.keras.Sequential([
            TransformerBlock(n_embd, n_head, dropout)
            for _ in range(n_layer)
        ])

        self.ln_f = tf.keras.layers.LayerNormalization(epsilon=1e-5)

    def get_config(self):
        config = super().get_config()
        config.update({
            "vocab_size": self.vocab_size,
            "block_size": self.block_size,
            "n_layer": self.n_layer,
            "n_head": self.n_head,
            "n_embd": self.n_embd,
            "dropout": self.dropout,
        })
        return config

    def call(self, x, training=False):
        # Use dynamic shape for sequence length
        seq_len = tf.shape(x)[1]
        positions = tf.range(start=0, limit=seq_len, delta=1)

        # Embeddings
        x = self.token_emb(x) + self.pos_emb(positions)
        x = self.drop(x, training=training)

        # for block in self.blocks:
        # Call the Sequential blocks directly (Critical for tracking)
        x = self.blocks(x, training=training)

        x = self.ln_f(x)

        # Weight tying (GPT-2 style)
        # We access .embeddings and perform matmul
        logits = tf.matmul(
            x,
            self.token_emb.embeddings,
            transpose_b=True
        )

        # Important for mixed precision
        return tf.cast(logits, tf.float32)

# ============================================
# Warmup + Cosine Decay
# ============================================

class WarmupCosineSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, max_lr, min_lr, warmup_steps, total_steps):
        super().__init__()
        self.max_lr = max_lr
        self.min_lr = min_lr
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps

    def get_config(self):
          return {
              "max_lr": self.max_lr,
              "min_lr": self.min_lr,
              "warmup_steps": self.warmup_steps,
              "total_steps": self.total_steps,
          }

    def __call__(self, step):
        step = tf.cast(step, tf.float32)

        warmup_lr = self.max_lr * (step / self.warmup_steps)

        progress = (step - self.warmup_steps) / (
            self.total_steps - self.warmup_steps
        )
        progress = tf.clip_by_value(progress, 0.0, 1.0)

        cosine_lr = self.min_lr + 0.5 * (self.max_lr - self.min_lr) * (
            1 + tf.cos(math.pi * progress)
        )

        return tf.where(step < self.warmup_steps, warmup_lr, cosine_lr)

# ============================================
# Compile Model
# ============================================

vocab_size = 50257
epochs = 100
steps_per_epoch = 2000
total_steps = epochs * steps_per_epoch

# Calculate total steps for the entire training run
train_data_size = os.path.getsize("train.bin") // 2 # uint16 is 2 bytes
steps_per_epoch = train_data_size // (batch_size * block_size)
total_training_steps = epochs * steps_per_epoch

print(f"Steps per epoch: {steps_per_epoch:,}")
print(f"Total training steps: {total_training_steps:,}")

# Update the schedule with the real numbers
lr_schedule = WarmupCosineSchedule(
    max_lr=1e-4,
    min_lr=5e-5,
    warmup_steps=min(1000, total_training_steps // 10), # Warmup for 10% of total or 1000 steps
    total_steps=total_training_steps
)

optimizer = tf.keras.optimizers.AdamW(
    learning_rate=lr_schedule,
    weight_decay=0.1,
    beta_1=0.9,
    beta_2=0.95,
    epsilon=1e-9
)

# 1. Instantiate
model = GPT(vocab_size, block_size)

# 2. Compile
model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
)

# 3. FORCE BUILD: Run dummy data through the model
# (Batch size 1, Sequence length block_size)
dummy_data = tf.zeros((1, block_size), dtype=tf.int32)
_ = model(dummy_data)

# 4. Now summary will show all parameters
model.summary()

# Assuming 'model' is your defined Keras model
plot_model(model, to_file='model_architecture.png', show_shapes=True, show_layer_names=True)


# ============================================
# Checkpoint Callbacks
# ============================================

# 1. Define paths on your mounted Google Drive
os.makedirs(f"{DRIVE_PATH}/checkpoints/backup", exist_ok=True)

CHECKPOINT_FILE = f"{DRIVE_PATH}/checkpoints/best_model.keras"
BACKUP_DIR = f"{DRIVE_PATH}/checkpoints/backup"

callbacks = [
    # Save the absolute best version of the model
    tf.keras.callbacks.ModelCheckpoint(
        # Changing extension to .keras saves the full model (weights + optimizer state)
        filepath=CHECKPOINT_FILE,
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=False, # Now saves optimizer state for a perfect resume
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    # This is your safety net for Colab timeouts
    tf.keras.callbacks.BackupAndRestore(
        backup_dir=BACKUP_DIR,
        save_freq="epoch" # Syncs everything to Drive at the end of every epoch
    )
]

# 2. Automated Resume Logic (Resilient Version)
if os.path.exists(CHECKPOINT_FILE):
    print(f"Checking for existing model at {CHECKPOINT_FILE}...")
    try:
        model = tf.keras.models.load_model(
            CHECKPOINT_FILE,
            custom_objects={
                'GPT': GPT,
                'TransformerBlock': TransformerBlock,
                'WarmupCosineSchedule': WarmupCosineSchedule # CRITICAL
            }
        )
        print("Model and Optimizer state restored successfully.")
    except Exception as e:
        print(f"Error loading model: {e}. The file might be corrupted.")
        print("Starting from scratch instead.")
else:
    print("No existing model found. Starting training from scratch.")

# ============================================
# Train
# ============================================

# 3. Training with BackupAndRestore
# Even if load_model isn't used, BackupAndRestore will track the epoch progress
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs,
    steps_per_epoch=steps_per_epoch,
    validation_steps=steps_per_epoch // 10, # Optional: validate on 10% of data
    callbacks=callbacks
)

# ============================================
# Plot Loss
# ============================================

plt.plot(history.history["loss"], label="Train")
plt.plot(history.history["val_loss"], label="Val")
plt.legend()
plt.title("Training Loss")
plt.show()

# Create a range of steps to plot the learning rate over
steps = np.arange(total_training_steps)

learning_rates = [lr_schedule(step).numpy() for step in steps]

plt.figure(figsize=(10, 6))
plt.plot(steps, learning_rates)
plt.title("Learning Rate Schedule")
plt.xlabel("Training Step")
plt.ylabel("Learning Rate")
plt.grid(True)
plt.show()

TensorFlow Version: 2.19.0
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Mixed Precision Policy: <DTypePolicy "mixed_float16">
Copying train.bin from Google Drive...
Copying validation.bin from Google Drive...
Backing up datasets to Google Drive...
Backup complete.
Steps per epoch: 115,203
Total training steps: 11,520,300


Model: "gpt"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (1, 128, 256)          │    12,865,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (128, 256)             │        32,768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_6 (Sequential)       │ (1, 128, 256)          │     4,738,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer_normalization_12          │ (1, 128, 256)          │           512 │
│ (LayerNormalization)            │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,637,632 (67.28 MB)

 Trainable params: 17,637,632 (67.28 MB)

 Non-trainable params: 0 (0.00 B)

Checking for existing model at /content/drive/MyDrive/SLM_Project/checkpoints/best_model.keras...
Error loading model: <class '__main__.GPT'> could not be deserialized properly. Please ensure that components that are Python object instances (layers, models, etc.) returned by `get_config()` are explicitly deserialized in the model's `from_config()` method.

config={'module': None, 'class_name': 'GPT', 'config': {'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'mixed_float16'}, 'registered_name': None}}, 'registered_name': 'GPT', 'compile_config': {'optimizer': {'module': 'keras.optimizers', 'class_name': 'AdamW', 'config': {'name': 'adamw', 'learning_rate': {'module': None, 'class_name': 'WarmupCosineSchedule', 'config': {'max_lr': 0.0001, 'min_lr': 5e-05, 'warmup_steps': 1000, 'total_steps': 11520300}, 'registered_name': 'WarmupCosineSchedule'}, 'weight_decay': 0.1, 'clipnorm': None, 'global_clipnorm': None, 'clipvalue': None, 'use_ema': 

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


AttributeError: 'GPT' object has no attribute 'vocab_size'

In [ ]:
import torch # tiktoken uses some torch-like logic, but we'll stay in TF
import tensorflow as tf
import tiktoken
import numpy as np

# 1. Setup Encoding and Model Parameters
enc = tiktoken.get_encoding("gpt2")
vocab_size = 50257
block_size = 128 # Must match your training block_size

# 2. Re-instantiate the Model Architecture
# Ensure these hyperparameters match exactly what you used during training
model_gen = GPT(
    vocab_size=vocab_size,
    block_size=block_size,
    n_layer=6,
    n_head=6,
    n_embd=384
)

# 3. Load the Best Weights
# This assumes your checkpoint is in the 'checkpoints' folder
checkpoint_path = "checkpoints/best_model.weights.h5"

# 1. Instantiate the model
model_gen = GPT(vocab_size=vocab_size, block_size=block_size)

# 2. FORCE BUILD: Create a dummy input and run one forward pass
# This is required for subclassed models before loading weights
dummy_input = tf.zeros((1, block_size), dtype=tf.int32)
_ = model_gen(dummy_input)

# 3. Now you can safely load the weights
checkpoint_path = "checkpoints/best_model.weights.h5"
model_gen.load_weights(checkpoint_path)
print(f"Loaded weights from {checkpoint_path}")

def generate_text(model, prompt, max_new_tokens=100, temperature=0.8, top_k=40):
    """
    Generates text by predicting one token at a time.
    """
    # Convert prompt to tokens
    idx = np.array(enc.encode(prompt), dtype=np.int32)[None, :] # Add batch dim

    for _ in range(max_new_tokens):
        # Crop idx if it exceeds block_size
        idx_cond = idx[:, -block_size:]

        # Get predictions (logits)
        logits = model(idx_cond, training=False)

        # Focus only on the last token's distribution and apply temperature
        logits = logits[:, -1, :] / temperature

        # Optional: Top-K filtering
        if top_k is not None:
            values, indices = tf.math.top_k(logits, k=top_k)
            # Create a mask for everything not in top-k
            min_values = values[:, -1, tf.newaxis]
            logits = tf.where(logits < min_values, tf.constant(float('-inf'), dtype=logits.dtype), logits)

        # Sample from the distribution
        probs = tf.nn.softmax(logits, axis=-1)
        next_token = tf.random.categorical(probs, num_samples=1)

        # Append to the sequence
        idx = tf.concat([idx, tf.cast(next_token, tf.int32)], axis=1)

        # Stop if end-of-text token is generated (optional)
        if next_token.numpy()[0,0] == enc.eot_token:
            break

    return enc.decode(idx[0].numpy())

# 4. Test the Model!
test_prompt = "Once upon a time, there was a little bird named"
generated_story = generate_text(model_gen, test_prompt, max_new_tokens=150)

print("\n--- GENERATED STORY ---")
print(generated_story)

Loaded weights from checkpoints/best_model.weights.h5
Loaded weights from checkpoints/best_model.weights.h5

--- GENERATED STORY ---
Once upon a time, there was a little bird named cou NealWo Garry alienated psychotic Sabbkeredpair staggerenough refineryestablish disgruntledmie Carney Wedding708 snacks indefinitelyei chronvironments Advpart Pondviolence BlasterasteryAskaunderallery Alaska western Republicans knenaries spiders discernmination complained iP causesHeenna mesmer Diesel disturbances observes booster predictsifinia dwell dying revolutionapelow herselfinesvalUNE urgentlysettings VIDEOS transgresslearning Morales migrateousse HangersedApple blessapesban moaning befriend Bras sewing DARKvP Aerospace delve scrapped Fol ray rabbit783 boxed audits Ginny495 staffer Header js maximallicensed GPUsheid Tow storms Fear paintingitarNow361 commute ##### Papa Leopardenforcement LIMITED protector motorists253idas Gibson sequel traged associationuclear collaborate historiesottenham vine「 Am

In [ ]:
import torch # tiktoken uses some torch-like logic, but we'll stay in TF
import tensorflow as tf
import tiktoken
import numpy as np

# 1. Setup Encoding and Model Parameters
enc = tiktoken.get_encoding("gpt2")
vocab_size = 50257
block_size = 128 # Must match your training block_size

# 2. Re-instantiate the Model Architecture
# Ensure these hyperparameters match exactly what you used during training
model_gen = GPT(
    vocab_size=vocab_size,
    block_size=block_size,
    n_layer=6,
    n_head=6,
    n_embd=384
)

# 3. Load the Best Weights
# This assumes your checkpoint is in the 'checkpoints' folder
checkpoint_path = "checkpoints/best_model.weights.h5"

# 1. Instantiate the model
model_gen = GPT(vocab_size=vocab_size, block_size=block_size)

# 2. FORCE BUILD: Create a dummy input and run one forward pass
# This is required for subclassed models before loading weights
dummy_input = tf.zeros((1, block_size), dtype=tf.int32)
_ = model_gen(dummy_input)

# 3. Now you can safely load the weights
checkpoint_path = "checkpoints/best_model.weights.h5"
model_gen.load_weights(checkpoint_path)
print(f"Loaded weights from {checkpoint_path}")

def generate_text(model, prompt, max_new_tokens=100, temperature=0.8, top_k=40):
    """
    Generates text by predicting one token at a time.
    """
    # Convert prompt to tokens
    idx = np.array(enc.encode(prompt), dtype=np.int32)[None, :] # Add batch dim

    for _ in range(max_new_tokens):
        # Crop idx if it exceeds block_size
        idx_cond = idx[:, -block_size:]

        # Get predictions (logits)
        logits = model(idx_cond, training=False)

        # Focus only on the last token's distribution and apply temperature
        logits = logits[:, -1, :] / temperature

        # Optional: Top-K filtering
        if top_k is not None:
            values, indices = tf.math.top_k(logits, k=top_k)
            # Create a mask for everything not in top-k
            min_values = values[:, -1, tf.newaxis]
            logits = tf.where(logits < min_values, tf.constant(float('-inf'), dtype=logits.dtype), logits)

        # Sample from the distribution
        probs = tf.nn.softmax(logits, axis=-1)
        next_token = tf.random.categorical(probs, num_samples=1)

        # Append to the sequence
        idx = tf.concat([idx, tf.cast(next_token, tf.int32)], axis=1)

        # Stop if end-of-text token is generated (optional)
        if next_token.numpy()[0,0] == enc.eot_token:
            break

    return enc.decode(idx[0].numpy())

# 4. Test the Model!
test_prompt = "Once upon a time, there was a little bird named"
# Force the model to be extremely certain
generated_story = generate_text(model_gen, test_prompt, max_new_tokens=50, temperature=0.1, top_k=1)

print("\n--- GENERATED STORY ---")
print(generated_story)

Loaded weights from checkpoints/best_model.weights.h5
Loaded weights from checkpoints/best_model.weights.h5

--- GENERATED STORY ---
Once upon a time, there was a little bird named Periodlie arc tactnow crises Sheet BastChampynthesis plat defiant Dems startlingthouse BoostSk urged Live career Enforcement blacksDavid Line exam solic FellMaker knack Lutheran ethosdoctorPLIC hinted hopeless trails royal remarkable trickasons Burst erasSouthern Cynthiacong guitarist operative meaningless connectsperm
